In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from utils import parse_har_files_to_dataframe
from common import BASE_HAR_DIR


sns.set_theme(style="whitegrid", context="paper")
pd.set_option('display.max_colwidth', 128)

df = parse_har_files_to_dataframe(BASE_HAR_DIR, test=False)


# SST - Detection method
- We use requests initiated by Google Tag Manager script and find that they always contain the same 36 URL parameters
- We then search for these parameters in requests not initiaed by a script from googletagmanager.com
- (TODO) Analyzing the scripts that initiated these requests (containing the 36 params), we find that they are always initiated by a first-party GTM script. We detect first-party GTM scripts by analyzing their content.

In [2]:
df['param_names_str'] = df['param_names'].apply(lambda x: ', '.join(x))

In [3]:
# list rows where the initiator domain contains googletagmanager.com
# params include gtm and the (request) entity is not Google
df_gtm = df[df['initiator_domains'].apply(lambda x: 'googletagmanager.com' in x)]
df_gtm = df_gtm[df_gtm.param_names_str.str.contains('gtm') & (df_gtm.entity != 'Google')]
df_gtm[['site_hostname', 'exp_mode', 'consent_mode', 'initiator_domains', 'req_hostname', 'param_names_str']]

,site_hostname,exp_mode,consent_mode,initiator_domains,req_hostname,param_names_str
1857,www.shop-apotheke.com,tracking,Reject,"[googletagmanager.com, sa-tech.de]",klpoz.shop-apotheke.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, gdid, cid, ecid, ul, sr, _fplc, ur, uaa, uab, uafvl, uamb, uam, uap, ..."
1885,www.shop-apotheke.com,tracking,Reject,"[googletagmanager.com, sa-tech.de]",klpoz.shop-apotheke.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, gdid, cid, ecid, ul, sr, _fplc, ur, uaa, uab, uafvl, uamb, uam, uap, ..."
2156,www.shop-apotheke.com,tracking,Reject,"[googletagmanager.com, sa-tech.de]",klpoz.shop-apotheke.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, ..."
2157,www.shop-apotheke.com,tracking,Reject,"[googletagmanager.com, sa-tech.de]",klpoz.shop-apotheke.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, ..."
2158,www.shop-apotheke.com,tracking,Reject,"[googletagmanager.com, sa-tech.de]",klpoz.shop-apotheke.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, ..."
...,...,...,...,...,...,...
58145,www.efarma.com,social,Accept,[googletagmanager.com],gtm.efarma.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, gdid, cid, ecid, ul, sr, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, u..."
58146,www.efarma.com,social,Accept,[googletagmanager.com],gtm.efarma.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, gdid, cid, ecid, ul, sr, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, u..."
58374,www.efarma.com,social,Accept,[googletagmanager.com],gtm.efarma.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, gdid, cid, ecid, ul, sr, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, u..."
58634,www.efarma.com,social,Accept,[googletagmanager.com],gtm.efarma.com,"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, gdid, cid, ecid, ul, sr, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, u..."


In [4]:
df_gtm.req_hostname.unique().tolist()

['klpoz.shop-apotheke.com',
 'pipeline.drogist.nl',
 'metrics.deonlinedrogist.nl',
 'sgtm.plein.nl',
 'ecom-data.trekpleister.nl',
 'ecom-data.kruidvat.nl',
 'v3-pixal-web.etos.nl',
 'sst.koopjesdrogisterij.nl',
 'inc.da.nl',
 'otasf.redcare.it',
 'gtm.efarma.com']

In [5]:
df_gtm['param_names_str'].value_counts().head(5).to_frame()

,count
param_names_str,
"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, uaw, are, pae, frm, pscdl, sst.rnd, sst.etld, sst.gcsub, sst.adr, sst.tft, sst.ude, dl, sid, sct, seg, dt, _s, tfd, richsstsse",26
"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, uaw, are, frm, pscdl, _eu, sst.rnd, sst.ngs, sst.etld, sst.gcsub, sst.tft, sst.ude, ngs, dl, dr, sid, sct, seg, dt, _s, tfd, richsstsse",25
"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, uaw, are, frm, pscdl, sst.rnd, sst.etld, sst.gcsub, sst.adr, sst.tft, sst.ude, dl, sid, sct, seg, dt, _s, tfd, richsstsse",18
"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, uaw, are, frm, pscdl, _eu, sst.rnd, sst.ngs, sst.etld, sst.gcsub, sst.tft, sst.ude, ngs, _s, dl, dr, sid, sct, seg, dt, en, ep.page_type, ep.event_id, ep.etos-tracking-consent, ep.x-fb-ck-fbp, ep.data_source, ep.login_status, ep.sitespect_id, ep.customer_group, ep.visitor_type, ep.customer_type, tfd, richsstsse",16
"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, uapv, uaw, are, pae, frm, pscdl, ec_mode, _eu, sst.rnd, sst.etld, sst.gcsub, sst.tft, sst.ude, _s, sid, sct, seg, dl, dr, dt, en, ep.page_hostname, ep.page_type, ep.login_state, ep.logged_in, ep.bc_segments, ep.cookie_consent, _et, ep.user_data.email, ep.user_data._tag_mode, up.user_cookie_consent, tfd, richsstsse",16


In [6]:
from functools import reduce

sets = df_gtm['param_names'].map(set)

# Step 2: Find the intersection of all sets
gtm_intersection = reduce(set.intersection, sets)

# Display the intersection
print(len(gtm_intersection), sorted(list(gtm_intersection)))

36 ['_p', '_s', 'are', 'cid', 'dl', 'dma', 'dma_cps', 'dt', 'ecid', 'frm', 'gcd', 'gcs', 'gtm', 'npa', 'pscdl', 'sct', 'seg', 'sid', 'sr', 'sst.rnd', 'sst.tft', 'sst.ude', 'tag_exp', 'tfd', 'tid', 'uaa', 'uab', 'uafvl', 'uam', 'uamb', 'uap', 'uapv', 'uaw', 'ul', 'ur', 'v']


In [20]:
df_ngtm = df[(df.param_names_str.str.contains('gtm')) &
             (df.param_names_str.str.contains('sr')) &
            #  (df.param_names_str.str.contains('dt')) &
            #  (df.param_names_str.str.contains('dl')) &
             (df.entity != 'Google')]
df_ngtm.req_hostname.value_counts().to_frame()

,count
req_hostname,
sgtm.plein.nl,226
ecom-data.trekpleister.nl,100
metrics.deonlinedrogist.nl,88
v3-pixal-web.etos.nl,87
ecom-data.kruidvat.nl,77
sst.koopjesdrogisterij.nl,68
pipeline.drogist.nl,67
klpoz.shop-apotheke.com,29
otasf.redcare.it,24


In [21]:
sets = df_ngtm['param_names'].map(set)

# Step 2: Find the intersection of all sets
ngtm_intersection = reduce(set.intersection, sets)

# Display the intersection
print(len(ngtm_intersection), sorted(list(ngtm_intersection)))

36 ['_p', '_s', 'are', 'cid', 'dl', 'dma', 'dma_cps', 'dt', 'ecid', 'frm', 'gcd', 'gcs', 'gtm', 'npa', 'pscdl', 'sct', 'seg', 'sid', 'sr', 'sst.rnd', 'sst.tft', 'sst.ude', 'tag_exp', 'tfd', 'tid', 'uaa', 'uab', 'uafvl', 'uam', 'uamb', 'uap', 'uapv', 'uaw', 'ul', 'ur', 'v']


In [25]:
df_ngtm[df_ngtm['initiator_domains'].apply(lambda x: 'googletagmanager.com' not in x)].initiator_urls


7589     [https://tmsst.aponeo.de/gtag/js?id=G-ZFZ1EWPSCZ&l=dataLayer&cx=c&sign=bc151c0673dd9b6c6906b7d1236246e70fe1705d5778c735c61e1...
7616     [https://tmsst.aponeo.de/gtag/js?id=G-ZFZ1EWPSCZ&l=dataLayer&cx=c&sign=bc151c0673dd9b6c6906b7d1236246e70fe1705d5778c735c61e1...
7647     [https://tmsst.aponeo.de/gtag/js?id=G-ZFZ1EWPSCZ&l=dataLayer&cx=c&sign=bc151c0673dd9b6c6906b7d1236246e70fe1705d5778c735c61e1...
7698     [https://www.aponeo.de/suche/?q=schwangerschaftstest, https://tmsst.aponeo.de/gtag/js?id=G-ZFZ1EWPSCZ&l=dataLayer&cx=c&sign=...
7707     [https://www.aponeo.de/suche/?q=schwangerschaftstest, https://tmsst.aponeo.de/gtag/js?id=G-ZFZ1EWPSCZ&l=dataLayer&cx=c&sign=...
                                                                      ...                                                               
61693    [https://sgtm.docpeter.it/gtmapwbued.js?id=G-JHTF9WM8WB&l=dataLayer&cx=c, https://www.docpeter.it/test-di-gravidanza-clearbl...
61808                                    

In [9]:
ngtm_intersection == gtm_intersection

True

In [10]:
expected_params = ngtm_intersection
GROUP_COLS = ['exp_mode', 'consent_mode', 'country', 'site_domain']

# ilter rows where 'param_names' contains all expected_params
df_all_gtm = df[df['param_names'].apply(lambda x: expected_params.issubset(x))]
df_all_gtm.drop_duplicates(['consent_mode', 'site_domain', 'req_hostname'])[
    ['exp_mode', 'consent_mode', 'country', 'site_domain', 'req_hostname']].\
        sort_values(['country', 'consent_mode', 'exp_mode'])

,exp_mode,consent_mode,country,site_domain,req_hostname
7589,tracking,Accept,DE,aponeo.de,tmsst.aponeo.de
10675,tracking,Accept,DE,shop-apotheke.com,klpoz.shop-apotheke.com
14406,tracking,Accept,DE,mycare.de,sgtm.mycare.de
1857,tracking,Reject,DE,shop-apotheke.com,klpoz.shop-apotheke.com
49517,social,Accept,IT,redcare.it,otasf.redcare.it
57337,social,Accept,IT,efarma.com,gtm.efarma.com
59266,social,Accept,IT,farmasave.it,sgtm.farmasave.it
61234,social,Accept,IT,docpeter.it,sgtm.docpeter.it
42789,social,Accept,NL,kruidvat.nl,ecom-data.kruidvat.nl
43579,social,Accept,NL,da.nl,inc.da.nl


In [11]:
sgtm_hosts = df_all_gtm.req_hostname.unique().tolist()
len(sgtm_hosts)

15

In [13]:
df_all_gtm[~df_all_gtm.url.str.contains('collect?v', regex=False)][
    ['site_domain', 'consent_mode', 'exp_mode', 'req_hostname', 'url']].\
        req_hostname.value_counts().to_frame()

,count
req_hostname,
sgtm.farmasave.it,22
sgtm.docpeter.it,8


In [14]:
df[
    df.param_names_str.str.contains('dl') &
    df.param_names_str.str.contains('dt') &
    df.param_names_str.str.contains('sr') &
    ~df.entity.isin(['Google']) &
    ~df['param_names'].apply(lambda x: expected_params.issubset(x))][
       ['site_domain', 'req_hostname', 'param_names_str', 'entity', 'initiator_domains']]

,site_domain,req_hostname,param_names_str,entity,initiator_domains


In [15]:
df_gtm[~df_gtm.param_names_str.str.contains('richsst')]

,site_hostname,exp_mode,country,consent_mode,url,initiator_type,initiator_urls,param_names,post_mime_type,post_data,req_domain,req_hostname,entity,site_domain,third_party,initiator_domains,param_names_str
36912,www.etos.nl,tracking,NL,Reject,https://v3-pixal-web.etos.nl/g/collect?v=2&tid=G-86YGNJXHJL&gtm=45je4a90v9106724000z878331478za200zb78331478&_p=172876985817...,script,"[https://www.etos.nl/cart/, https://www.googletagmanager.com/gtag/js?id=G-86YGNJXHJL&l=dataLayer&cx=c, https://www.googletag...","[v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, u...",,,etos.nl,v3-pixal-web.etos.nl,Ahold Delhaize,etos.nl,False,"[etos.nl, googletagmanager.com, googletagmanager.com]","v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, ua..."
36913,www.etos.nl,tracking,NL,Reject,https://v3-pixal-web.etos.nl/g/collect?v=2&tid=G-86YGNJXHJL&gtm=45je4a90v9106724000z878331478za200zb78331478&_p=172876985817...,script,"[https://www.etos.nl/cart/, https://www.googletagmanager.com/gtag/js?id=G-86YGNJXHJL&l=dataLayer&cx=c, https://www.googletag...","[v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, u...",,,etos.nl,v3-pixal-web.etos.nl,Ahold Delhaize,etos.nl,False,"[etos.nl, googletagmanager.com, googletagmanager.com]","v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, ua..."
36914,www.etos.nl,tracking,NL,Reject,https://v3-pixal-web.etos.nl/g/collect?v=2&tid=G-86YGNJXHJL&gtm=45je4a90v9106724000z878331478za200zb78331478&_p=172876985817...,script,"[https://www.etos.nl/cart/, https://www.googletagmanager.com/gtag/js?id=G-86YGNJXHJL&l=dataLayer&cx=c, https://www.googletag...","[v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, u...",,,etos.nl,v3-pixal-web.etos.nl,Ahold Delhaize,etos.nl,False,"[etos.nl, googletagmanager.com, googletagmanager.com]","v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, ua..."
36915,www.etos.nl,tracking,NL,Reject,https://v3-pixal-web.etos.nl/g/collect?v=2&tid=G-86YGNJXHJL&gtm=45je4a90v9106724000z878331478za200zb78331478&_p=172876985817...,script,"[https://www.etos.nl/cart/, https://www.googletagmanager.com/gtag/js?id=G-86YGNJXHJL&l=dataLayer&cx=c, https://www.googletag...","[v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, u...",,,etos.nl,v3-pixal-web.etos.nl,Ahold Delhaize,etos.nl,False,"[etos.nl, googletagmanager.com, googletagmanager.com]","v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, ua..."
45510,www.etos.nl,social,NL,Accept,https://v3-pixal-web.etos.nl/g/collect?v=2&tid=G-86YGNJXHJL&gtm=45je4a90v9106724000z878331478za200zb78331478&_p=172864645691...,script,[https://www.googletagmanager.com/gtag/js?id=G-86YGNJXHJL&l=dataLayer&cx=c],"[v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, u...",text/plain;charset=UTF-8,en=view_promotion&ep.page_type=home&ep.event_id=1728646457198.564947.5&ep.data_source=web&ep.login_status=0&ep.sitespect_id=...,etos.nl,v3-pixal-web.etos.nl,Ahold Delhaize,etos.nl,False,[googletagmanager.com],"v, tid, gtm, _p, gcs, gcd, npa, dma_cps, dma, tag_exp, cid, ecid, ul, sr, _fplc, ir, ur, uaa, uab, uafvl, uamb, uam, uap, ua..."
45701,www.etos.nl,social,NL,Accept,https://v3-pixal-web.etos.nl/g/collect?v=2&tid=G-86YGNJXHJL&gtm=45je4a90v9106724000z878331478za200zb78331478&_p=172864646029...,script,"[https://www.googletagmanager.com/gtag/destination?id=DC-9866847&l=dataLayer&cx=c, https://www.googletagmanager.com/gtag/js?...","[v, tid, gtm, _p, gcs, gcd, npa, dma_cps, 